# YOLO Vehicle Detection + Distance + Speed + TTC\n\nUpload a traffic video, run YOLO tracking, estimate distance/speed/TTC, and download an annotated output video.

In [ ]:
!pip -q install ultralytics opencv-python

In [ ]:
from google.colab import files\nuploaded = files.upload()\nvideo_path = next(iter(uploaded.keys()))\nvideo_path

In [ ]:
import math\nfrom collections import deque\nfrom dataclasses import dataclass, field\n\nimport cv2\nfrom ultralytics import YOLO\n\nVEHICLE_CLASSES = {'car', 'truck', 'bus', 'motorcycle', 'bicycle'}\nREAL_WIDTH_METERS = {'car': 1.8, 'truck': 2.6, 'bus': 2.7, 'motorcycle': 0.8, 'bicycle': 0.65}\n\n@dataclass\nclass TrackState:\n    track_id: int\n    class_name: str\n    distance_m: float\n    speed_mps: float = 0.0\n    ttc_s: float = math.inf\n    history: deque = field(default_factory=lambda: deque(maxlen=24))\n\ndef estimate_distance(box_width_px, class_name, focal_px=850.0):\n    real_width = REAL_WIDTH_METERS.get(class_name, 1.8)\n    return max(0.1, (real_width * focal_px) / max(box_width_px, 1.0))\n\ndef color_for(ttc):\n    if ttc < 2.5:\n        return (0, 0, 255)\n    if ttc < 5.0:\n        return (0, 215, 255)\n    return (40, 220, 110)\n\ndef draw_label(frame, text, x, y, color):\n    font = cv2.FONT_HERSHEY_SIMPLEX\n    scale = 0.55\n    thickness = 2\n    (w, h), _ = cv2.getTextSize(text, font, scale, thickness)\n    y = max(h + 8, y)\n    cv2.rectangle(frame, (x, y - h - 8), (x + w + 10, y + 4), (0, 0, 0), -1)\n    cv2.putText(frame, text, (x + 5, y - 3), font, scale, color, thickness, cv2.LINE_AA)

In [ ]:
model = YOLO('yolo11n.pt')\noutput_path = 'yolo_distance_ttc_output.mp4'\n\ncap = cv2.VideoCapture(video_path)\nfps = cap.get(cv2.CAP_PROP_FPS) or 30\nwidth = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))\nheight = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))\ncap.release()\n\nwriter = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))\ntracks = {}\n\nfor result in model.track(source=video_path, conf=0.35, persist=True, stream=True, verbose=False):\n    frame = result.orig_img.copy()\n    danger_active = False\n\n    if result.boxes is not None:\n        for box in result.boxes:\n            if box.id is None:\n                continue\n\n            class_id = int(box.cls[0])\n            class_name = model.names[class_id]\n            if class_name not in VEHICLE_CLASSES:\n                continue\n\n            track_id = int(box.id[0])\n            x1, y1, x2, y2 = [float(v) for v in box.xyxy[0]]\n            box_width = x2 - x1\n            distance_m = estimate_distance(box_width, class_name)\n\n            previous = tracks.get(track_id)\n            speed_mps = 0.0\n            ttc_s = math.inf\n            if previous:\n                closing_speed = max((previous.distance_m - distance_m) * fps, 0.0)\n                speed_mps = previous.speed_mps * 0.7 + closing_speed * 0.3\n                ttc_s = distance_m / speed_mps if speed_mps > 0.2 else math.inf\n\n            state = TrackState(track_id, class_name, distance_m, speed_mps, ttc_s)\n            state.history = previous.history if previous else deque(maxlen=24)\n            state.history.append(((x1 + x2) / 2, (y1 + y2) / 2))\n            tracks[track_id] = state\n\n            color = color_for(ttc_s)\n            danger_active = danger_active or ttc_s < 2.5\n            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)\n            speed_mph = speed_mps * 2.237\n            draw_label(frame, f'ID {track_id} {class_name} {speed_mph:.0f}mph', int(x1), int(y1) - 8, color)\n            if math.isfinite(ttc_s) and ttc_s < 5:\n                draw_label(frame, f'TTC {ttc_s:.1f}s | {distance_m:.0f}m', int(x1), int(y2) + 22, color)\n\n            points = list(state.history)\n            for p1, p2 in zip(points, points[1:]):\n                cv2.line(frame, tuple(map(int, p1)), tuple(map(int, p2)), color, 2)\n\n    if danger_active:\n        cv2.rectangle(frame, (0, height // 2 - 24), (width, height // 2 + 24), (0, 0, 180), -1)\n        cv2.putText(frame, '!! DANGER ALERT !!', (max(20, width // 2 - 180), height // 2 + 10), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255,255,255), 3, cv2.LINE_AA)\n\n    writer.write(frame)\n\nwriter.release()\noutput_path

In [ ]:
from google.colab import files\nfiles.download(output_path)